# Zaid Hybrid IDS — Step 7: Final Frozen Detector Test
This notebook performs the one-time final evaluation of the frozen seed-20260829 Random Forest detector without Dst Port at threshold 0.5. It loads the test partition only, performs no training or tuning, measures repeated batched inference time, and computes stratified bootstrap confidence intervals.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)

PROJECT_DIR = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_DIR / 'data' / 'splits'
RESULT_DIR = PROJECT_DIR / 'results'
MODEL_DIR = PROJECT_DIR / 'models'
TEST_PATH = SPLIT_DIR / 'test.parquet'
FEATURE_MANIFEST_PATH = RESULT_DIR / 'feature_sets_manifest.json'
SPLIT_MANIFEST_PATH = RESULT_DIR / 'grouped_split_manifest.json'
STABILITY_MANIFEST_PATH = RESULT_DIR / 'random_forest_no_dst_port_five_seed_manifest.json'
FROZEN_MODEL_PATH = MODEL_DIR / 'random_forest_no_dst_port.joblib'
SEED = 20260829
THRESHOLD = 0.5
BOOTSTRAP_REPLICATES = 500

for path in [TEST_PATH, FEATURE_MANIFEST_PATH, SPLIT_MANIFEST_PATH, STABILITY_MANIFEST_PATH, FROZEN_MODEL_PATH]:
    assert path.exists(), f'Missing required file: {path}'

with open(FEATURE_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    feature_manifest = json.load(file_handle)
with open(SPLIT_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    split_manifest = json.load(file_handle)
with open(STABILITY_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    stability_manifest = json.load(file_handle)

assert stability_manifest['stable'] is True
assert stability_manifest['frozen_detector_seed'] == SEED
assert stability_manifest['test_partition_loaded_or_used'] is False

print('FINAL TEST ACCESS: loading the held-out test partition for one-time evaluation.')
test_df = pd.read_parquet(TEST_PATH)
frozen_model = joblib.load(FROZEN_MODEL_PATH)
assert test_df.shape == (156717, 81), test_df.shape
print('Test:', test_df.shape)
print('Training performed in this notebook: False')
print('Threshold tuning performed in this notebook: False')

FINAL TEST ACCESS: loading the held-out test partition for one-time evaluation.
Test: (156717, 81)
Training performed in this notebook: False
Threshold tuning performed in this notebook: False


In [3]:
FEATURES = feature_manifest['no_dst_port_feature_set']
assert len(FEATURES) == 60
assert 'Dst Port' not in FEATURES
X_test = test_df[FEATURES].astype(np.float32)
y_test_original = test_df['Label'].astype(str).reset_index(drop=True)
y_test = y_test_original.ne('Benign').astype(np.int8).to_numpy()
assert np.isfinite(X_test.to_numpy(copy=False)).all()
assert y_test_original.value_counts().to_dict() == {
    'Benign': 99571, 'FTP-BruteForce': 29007, 'SSH-Bruteforce': 28139
}
print('Feature count:', len(FEATURES))
print('Test labels:', y_test_original.value_counts().to_dict())
print('Frozen test preparation: PASSED')

Feature count: 60
Test labels: {'Benign': 99571, 'FTP-BruteForce': 29007, 'SSH-Bruteforce': 28139}
Frozen test preparation: PASSED


In [4]:
# Warm up, then repeat full batched inference five times.
_ = frozen_model.predict_proba(X_test.iloc[:5000])[:, 1]
inference_times = []
test_probability = None
max_repeat_probability_difference = 0.0
for repeat in range(5):
    start = perf_counter()
    current_probability = frozen_model.predict_proba(X_test)[:, 1]
    elapsed = perf_counter() - start
    inference_times.append(elapsed)
    if test_probability is None:
        test_probability = current_probability
    else:
        repeat_difference = float(np.max(np.abs(test_probability - current_probability)))
        max_repeat_probability_difference = max(
            max_repeat_probability_difference, repeat_difference
        )
        assert np.allclose(
            test_probability, current_probability, rtol=1e-12, atol=1e-12
        )

test_prediction = (test_probability >= THRESHOLD).astype(np.int8)
tn, fp, fn, tp = confusion_matrix(y_test, test_prediction, labels=[0, 1]).ravel()
false_positive_rate = fp / (fp + tn) if (fp + tn) else 0.0
ftp_mask = y_test_original.eq('FTP-BruteForce').to_numpy()
ssh_mask = y_test_original.eq('SSH-Bruteforce').to_numpy()
median_inference_seconds = float(np.median(inference_times))

metrics = {
    'model': 'random_forest_no_dst_port',
    'frozen_seed': SEED, 'threshold': THRESHOLD,
    'test_rows': int(len(test_df)), 'feature_count': len(FEATURES),
    'accuracy': accuracy_score(y_test, test_prediction),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_prediction),
    'precision': precision_score(y_test, test_prediction, zero_division=0),
    'recall': recall_score(y_test, test_prediction, zero_division=0),
    'f1': f1_score(y_test, test_prediction, zero_division=0),
    'macro_f1': f1_score(y_test, test_prediction, average='macro'),
    'pr_auc': average_precision_score(y_test, test_probability),
    'roc_auc': roc_auc_score(y_test, test_probability),
    'brier_score': brier_score_loss(y_test, test_probability),
    'false_positive_rate': false_positive_rate,
    'ftp_attack_recall': float(test_prediction[ftp_mask].mean()),
    'ssh_attack_recall': float(test_prediction[ssh_mask].mean()),
    'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    'inference_repeats': len(inference_times),
    'inference_seconds_mean': float(np.mean(inference_times)),
    'inference_seconds_std': float(np.std(inference_times, ddof=1)),
    'inference_seconds_median': median_inference_seconds,
    'max_repeat_probability_difference': max_repeat_probability_difference,
    'batch_median_microseconds_per_flow': median_inference_seconds / len(test_df) * 1e6,
    'batch_median_throughput_flows_per_second': len(test_df) / median_inference_seconds,
    'model_size_mb': FROZEN_MODEL_PATH.stat().st_size / (1024 ** 2)
}
print(pd.Series(metrics).to_string())
print('Inference times:', inference_times)
print('FINAL TEST INFERENCE: COMPLETE')

model                                       random_forest_no_dst_port
frozen_seed                                                  20260829
threshold                                                         0.5
test_rows                                                      156717
feature_count                                                      60
accuracy                                                     0.999974
balanced_accuracy                                            0.999972
precision                                                    0.999965
recall                                                       0.999965
f1                                                           0.999965
macro_f1                                                     0.999972
pr_auc                                                            1.0
roc_auc                                                           1.0
brier_score                                                  0.000017
false_positive_rate 

In [5]:
# Stratified nonparametric bootstrap for 95% confidence intervals.
rng = np.random.default_rng(SEED)
benign_indices = np.flatnonzero(y_test == 0)
attack_indices = np.flatnonzero(y_test == 1)
bootstrap_records = []
for replicate in range(BOOTSTRAP_REPLICATES):
    sampled_indices = np.concatenate([
        rng.choice(benign_indices, size=len(benign_indices), replace=True),
        rng.choice(attack_indices, size=len(attack_indices), replace=True)
    ])
    sampled_true = y_test[sampled_indices]
    sampled_prediction = test_prediction[sampled_indices]
    sampled_tn, sampled_fp, sampled_fn, sampled_tp = confusion_matrix(
        sampled_true, sampled_prediction, labels=[0, 1]
    ).ravel()
    bootstrap_records.append({
        'accuracy': accuracy_score(sampled_true, sampled_prediction),
        'precision': precision_score(sampled_true, sampled_prediction, zero_division=0),
        'recall': recall_score(sampled_true, sampled_prediction, zero_division=0),
        'f1': f1_score(sampled_true, sampled_prediction, zero_division=0),
        'macro_f1': f1_score(sampled_true, sampled_prediction, average='macro'),
        'false_positive_rate': sampled_fp / (sampled_fp + sampled_tn)
    })
bootstrap_df = pd.DataFrame(bootstrap_records)
confidence_rows = []
for metric_name in bootstrap_df.columns:
    confidence_rows.append({
        'metric': metric_name,
        'estimate': float(metrics[metric_name]),
        'bootstrap_mean': float(bootstrap_df[metric_name].mean()),
        'ci_2_5_percent': float(bootstrap_df[metric_name].quantile(0.025)),
        'ci_97_5_percent': float(bootstrap_df[metric_name].quantile(0.975)),
        'replicates': BOOTSTRAP_REPLICATES
    })
confidence_intervals = pd.DataFrame(confidence_rows)
print(confidence_intervals.to_string(index=False))
print('BOOTSTRAP CONFIDENCE INTERVALS: COMPLETE')

             metric  estimate  bootstrap_mean  ci_2_5_percent  ci_97_5_percent  replicates
           accuracy  0.999974        0.999974        0.999943         0.999994         500
          precision  0.999965        0.999964        0.999913         1.000000         500
             recall  0.999965        0.999964        0.999895         1.000000         500
                 f1  0.999965        0.999964        0.999921         0.999991         500
           macro_f1  0.999972        0.999972        0.999938         0.999993         500
false_positive_rate  0.000020        0.000021        0.000000         0.000050         500
BOOTSTRAP CONFIDENCE INTERVALS: COMPLETE


In [6]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

predictions = pd.DataFrame({
    'test_row': np.arange(len(test_df), dtype=np.int64),
    'flow_group_id': test_df['flow_group_id'].to_numpy(),
    'original_label': y_test_original,
    'binary_label': y_test,
    'attack_probability': test_probability.astype(np.float32),
    'prediction': test_prediction,
    'correct': test_prediction == y_test
})
error_mask = predictions['correct'].eq(False)
misclassifications = predictions.loc[error_mask].copy()
misclassifications['error_type'] = np.where(
    misclassifications['binary_label'].eq(0), 'false_positive', 'false_negative'
)
test_context = test_df.reset_index(drop=True)[['Timestamp', 'Dst Port', 'Protocol']].copy()
test_context.insert(0, 'test_row', np.arange(len(test_context), dtype=np.int64))
misclassifications = misclassifications.merge(
    test_context, on='test_row', how='left', validate='one_to_one'
)

confusion = pd.DataFrame(
    [[int(tn), int(fp)], [int(fn), int(tp)]],
    index=['Actual Benign', 'Actual Attack'],
    columns=['Predicted Benign', 'Predicted Attack']
)

METRICS_PATH = RESULT_DIR / 'final_frozen_test_metrics.csv'
CONFUSION_PATH = RESULT_DIR / 'final_frozen_test_confusion_matrix.csv'
PREDICTIONS_PATH = RESULT_DIR / 'final_frozen_test_predictions.parquet'
MISCLASSIFICATIONS_PATH = RESULT_DIR / 'final_frozen_test_misclassifications.csv'
CI_PATH = RESULT_DIR / 'final_frozen_test_bootstrap_ci.csv'
MANIFEST_PATH = RESULT_DIR / 'final_frozen_test_manifest.json'
pd.DataFrame([metrics]).to_csv(METRICS_PATH, index=False)
confusion.to_csv(CONFUSION_PATH)
predictions.to_parquet(PREDICTIONS_PATH, index=False, compression='zstd')
misclassifications.to_csv(MISCLASSIFICATIONS_PATH, index=False)
confidence_intervals.to_csv(CI_PATH, index=False)

test_output_metadata = split_manifest['outputs']['test']
manifest = {
    'stage': 'one_time_final_frozen_test',
    'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': 'random_forest_no_dst_port',
    'model_sha256': sha256_file(FROZEN_MODEL_PATH),
    'frozen_seed': SEED, 'threshold': THRESHOLD,
    'feature_set': 'no_dst_port_feature_set',
    'feature_count': len(FEATURES),
    'test_file': TEST_PATH.name,
    'test_file_sha256': test_output_metadata['sha256'],
    'test_rows': int(len(test_df)),
    'training_performed_in_this_notebook': False,
    'threshold_or_hyperparameter_tuning_on_test': False,
    'bootstrap_method': '500-replicate stratified nonparametric bootstrap',
    'post_test_policy': 'Do not change model, features, seed, or threshold based on final test results',
    'latency_note': 'Batched inference measurement; per-flow and streaming latency are evaluated separately',
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'python_version': sys.version,
    'platform': platform.platform()
}
with open(MANIFEST_PATH, 'w', encoding='utf-8') as file_handle:
    json.dump(manifest, file_handle, ensure_ascii=False, indent=2)

print('Final metrics:')
print(pd.Series(metrics).to_string())
print('\nConfusion matrix:')
print(confusion.to_string())
print('\nMisclassified rows:', len(misclassifications))
print('Saved:', METRICS_PATH)
print('Saved:', CONFUSION_PATH)
print('Saved:', PREDICTIONS_PATH)
print('Saved:', MISCLASSIFICATIONS_PATH)
print('Saved:', CI_PATH)
print('Saved:', MANIFEST_PATH)
print('FINAL FROZEN TEST PASSED: evaluation saved; model configuration remains frozen.')

Final metrics:
model                                       random_forest_no_dst_port
frozen_seed                                                  20260829
threshold                                                         0.5
test_rows                                                      156717
feature_count                                                      60
accuracy                                                     0.999974
balanced_accuracy                                            0.999972
precision                                                    0.999965
recall                                                       0.999965
f1                                                           0.999965
macro_f1                                                     0.999972
pr_auc                                                            1.0
roc_auc                                                           1.0
brier_score                                                  0.000017
false